<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Фактуры:</h2>

----

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о
фактурах за поставленные товары или оказанные услуги. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
public class LineItem
{
    private string _name;
    private decimal _price;

    // Свойства с геттерами и сеттерами + проверка
    public string Name
    {
        get { return _name; }
        set { _name = string.IsNullOrWhiteSpace(value) ? "Без названия" : value; }
    }

    public decimal Price
    {
        get { return _price; }
        set { _price = value < 0 ? 0 : value; }
    }

    public LineItem(string name, decimal price)
    {
        Name = name;
        Price = price;
    }
}

public class Invoice
{
    private string _invoiceNumber;
    private DateTime _issueDate;
    private decimal _totalAmount;

    // Свойства с явными геттерами/сеттерами
    public string InvoiceNumber
    {
        get { return _invoiceNumber; }
        set { _invoiceNumber = string.IsNullOrWhiteSpace(value) ? "INV-000" : value; }
    }

    public DateTime IssueDate
    {
        get { return _issueDate; }
        set { _issueDate = value; }
    }

    public decimal TotalAmount
    {
        get { return _totalAmount; }
        protected set { _totalAmount = value < 0 ? 0 : value; }
    }

    protected List<LineItem> Items = new List<LineItem>();

    // Конструктор с использованием свойств
    public Invoice(string number, DateTime date)
    {
        InvoiceNumber = number;
        IssueDate = date;
    }

    // Виртуальный метод расчёта (Полиморфизм)
    public virtual decimal CalculateTotal()
    {
        _totalAmount = Items.Sum(item => item.Price);
        return _totalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        Items.Add(lineItem);
        Console.WriteLine($"[Счёт {InvoiceNumber}] Добавлена позиция: {lineItem.Name} — {lineItem.Price} руб.");
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null && Items.Remove(lineItem))
        {
            Console.WriteLine($"[Счёт {InvoiceNumber}] Удалена позиция: {lineItem.Name}");
        }
    }

    // =====================================================
    // ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: перенос позиции в другой счёт
    // =====================================================
    public void SendItemTo(LineItem item, Invoice target)
    {
        if (item == null || target == null) return;
        if (Items.Contains(item))
        {
            RemoveLine(item);
            target.AddLine(item);
            Console.WriteLine($"--> [Взаимодействие] '{item.Name}' перенесён из '{InvoiceNumber}' → '{target.InvoiceNumber}'");
        }
    }

    // =====================================================
    // ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: слияние другого счёта в текущий
    // =====================================================
    public void MergeWith(Invoice other)
    {
        if (other == null || other == this) return;
        Console.WriteLine($"\n--> [Взаимодействие] Слияние '{other.InvoiceNumber}' в '{this.InvoiceNumber}'...");
        // Копируем список, чтобы не изменять его во время обхода
        List<LineItem> toMove = new List<LineItem>(other.Items);
        foreach (LineItem line in toMove)
        {
            other.SendItemTo(line, this);
        }
    }
}

// 1. Товарная фактура
public class GoodsInvoice : Invoice
{
    private DateTime _supplyDate;

    public DateTime SupplyDate
    {
        get { return _supplyDate; }
        set { _supplyDate = value; }
    }

    public GoodsInvoice(string number, DateTime date, DateTime supplyDate) : base(number, date)
    {
        SupplyDate = supplyDate;
    }

    // Переопределение метода (Полиморфизм)
    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine("--> Дата поставки товара: " + SupplyDate.ToShortDateString());
    }
}

// 2. Услуговая фактура
public class ServiceInvoice : Invoice
{
    private DateTime _serviceDate;

    public DateTime ServiceDate
    {
        get { return _serviceDate; }
        set { _serviceDate = value; }
    }

    public ServiceInvoice(string number, DateTime date, DateTime serviceDate) : base(number, date)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("--> Причина удаления: Услуга была отменена клиентом.");
    }
}

// 3. Комбинированная фактура
public class CombinedInvoice : Invoice
{
    private bool _returnAllowed;

    public bool ReturnAllowed
    {
        get { return _returnAllowed; }
        set { _returnAllowed = value; }
    }

    public CombinedInvoice(string number, DateTime date, bool returnAllowed) : base(number, date)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();
        if (ReturnAllowed)
        {
            total = total + (total * 0.05m); // +5%
            Console.WriteLine("(Включен сбор за возможность возврата 5%)");
        }
        return total;
    }
}

// ==========================================
// ПРОВЕРКА И ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ
// ==========================================
Console.WriteLine("=== Товарная фактура (GoodsInvoice) ===");
GoodsInvoice goods = new GoodsInvoice("Т-001", DateTime.Now, DateTime.Now.AddDays(3));
goods.AddLine(new LineItem("Ноутбук", 50000));
goods.AddLine(new LineItem("Мышь", 1200));

Console.WriteLine("\n=== Услуговая фактура (ServiceInvoice) ===");
ServiceInvoice service = new ServiceInvoice("У-001", DateTime.Now, DateTime.Now.AddDays(1));
LineItem consulting = new LineItem("Консультация", 8000);
service.AddLine(consulting);

Console.WriteLine("\n=== Удаление позиции в услуговой фактуре ===");
service.RemoveLine(consulting);

Console.WriteLine("\n=== Комбинированная фактура (CombinedInvoice) ===");
CombinedInvoice combined = new CombinedInvoice("К-001", DateTime.Now, true);
combined.AddLine(new LineItem("Смартфон", 30000));
combined.AddLine(new LineItem("Настройка ПО", 2500));
decimal total = combined.CalculateTotal();
Console.WriteLine($"Итоговая сумма: {total} руб.");

Console.WriteLine("\n=== ВЗАИМОДЕЙСТВИЕ: Перенос позиции из GoodsInvoice в ServiceInvoice ===");
LineItem mouse = new LineItem("Мышь", 1200);
goods.AddLine(mouse); // добавим отдельно для примера переноса
goods.SendItemTo(mouse, service);

Console.WriteLine("\n=== ВЗАИМОДЕЙСТВИЕ: Слияние ServiceInvoice в CombinedInvoice ===");
combined.MergeWith(service);

Console.WriteLine($"\n[Итог] Комбинированный счёт после слияния: {combined.CalculateTotal()} руб.");

=== Товарная фактура (GoodsInvoice) ===
[Счёт Т-001] Добавлена позиция: Ноутбук — 50000 руб.
--> Дата поставки товара: 23.09.2026
[Счёт Т-001] Добавлена позиция: Мышь — 1200 руб.
--> Дата поставки товара: 23.09.2026

=== Услуговая фактура (ServiceInvoice) ===
[Счёт У-001] Добавлена позиция: Консультация — 8000 руб.

=== Удаление позиции в услуговой фактуре ===
[Счёт У-001] Удалена позиция: Консультация
--> Причина удаления: Услуга была отменена клиентом.

=== Комбинированная фактура (CombinedInvoice) ===
[Счёт К-001] Добавлена позиция: Смартфон — 30000 руб.
[Счёт К-001] Добавлена позиция: Настройка ПО — 2500 руб.
(Включен сбор за возможность возврата 5%)
Итоговая сумма: 34125,00 руб.

=== ВЗАИМОДЕЙСТВИЕ: Перенос позиции из GoodsInvoice в ServiceInvoice ===
[Счёт Т-001] Добавлена позиция: Мышь — 1200 руб.
--> Дата поставки товара: 23.09.2026
[Счёт Т-001] Удалена позиция: Мышь
[Счёт У-001] Добавлена позиция: Мышь — 1200 руб.
--> [Взаимодействие] 'Мышь' перенесён из 'Т-001' → 'У-001'

===